# Varlen Gated DeltaNet v2 (SM75 Turing / Tesla T4) — Build, Verify, Benchmark & Nsight Compute

End-to-end guide on **NVIDIA Tesla T4 (SM75)** (Google Colab / Kaggle / local CUDA GPU):
1. Verify GPU, CUDA, PyTorch environment, initialize **NVIDIA CUTLASS**, and install open-source baselines.
2. Build the CUDA/CuTe extension and package a `.whl` into `wheels/`.
3. Verify numerical correctness (Varlen Fwd/Bwd, Decode `L=1`, E2E Prefill->Decode Handoff, CUDAGraph, and `torch.compile`).
4. Run **Training Benchmark** (Token Mixer vs Baselines + E2E `DeltaBlock` with `torch.compile`) and save `assets/training_benchmark.png`.
5. Run **Inference Benchmark** (Decode `L=1` & E2E Prefill+Generation vs Baselines & `torch.compile`) and save `assets/inference_benchmark.png`.
6. Run **NVIDIA Nsight Compute (`ncu`)** to export isolated `.ncu-rep` reports (1 kernel per file, multi-case per file) into `benchmark/`.


## 1. Check GPU & Prepare Source / CUTLASS (+ Baselines)


In [ ]:
import os, sys

if not os.path.isfile("setup.py") and not os.path.isdir("Varlen-Gated-DeltaNet"):
    !git clone --recursive --depth 1 https://github.com/CBG6682/Varlen-Gated-DeltaNet.git

if os.path.isdir("Varlen-Gated-DeltaNet"):
    %cd Varlen-Gated-DeltaNet
elif os.path.basename(os.getcwd()) == "benchmark":
    %cd ..

print("Working directory:", os.getcwd())

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA runtime:", torch.version.cuda)
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0), "| Capability:", torch.cuda.get_device_capability(0))

!nvidia-smi


In [ ]:
%%bash
set -e
if [ ! -f "third_party/cutlass/include/cute/tensor.hpp" ]; then
    echo "Initializing NVIDIA CUTLASS submodule in third_party/cutlass..."
    git submodule update --init --depth 1 third_party/cutlass
else
    echo "CUTLASS headers already exist in third_party/cutlass/include."
fi

# Install open-source baselines for Section 4 & 5 comparisons
pip install -q einops
pip install -q git+https://github.com/sustcsonglin/flash-linear-attention.git@2705ab8ee9a9086bcfa6d079858e4281e01215a1 --no-build-isolation || true
if [ ! -d "/tmp/gateddeltanet-2" ]; then
    git clone -q https://github.com/NVlabs/GatedDeltaNet-2.git /tmp/gateddeltanet-2
    (cd /tmp/gateddeltanet-2 && git checkout -q de8476de0fecae08fc16f0bf3154900244c61c73)
    echo "# Standalone" > /tmp/gateddeltanet-2/lit_gpt/__init__.py
fi


## 2. Build Extension & Package Wheel (`wheels/`)
Builds `_C*.so` in-tree, overwrites the `.whl` in `wheels/`, and installs in editable mode. If a matching `.whl` already exists and you do not want to rebuild, stop/skip this cell and run `pip install --no-deps wheels/*.whl` instead.


In [ ]:
%%bash
set -euo pipefail
export MAX_JOBS=2
export VERBOSE=1
export TORCH_EXTENSIONS_VERBOSE=1

pip install -q ninja wheel setuptools

# Clean previous build outputs and rebuild in-tree extension cleanly
rm -rf build dist deltanet_v2/_C*.so
FORCE_REBUILD=1 python setup.py build_ext --inplace -v

# Package and overwrite prebuilt wheel inside wheels/
rm -f wheels/*.whl
FORCE_REBUILD=0 python -m pip wheel -v --no-deps --no-build-isolation -w wheels .

# Install package in editable mode pointing to current source
FORCE_REBUILD=0 python -m pip install -v --no-deps --no-build-isolation -e .
ls -lh wheels/*.whl deltanet_v2/_C*.so


## 3. Correctness Verification (Correctness + Autograd + `torch.compile`)
Verifies raw varlen kernels (`deltanet_fwd`, `deltanet_bwd`, zero-length edge cases), single-step decode (`L=1`, edge cases, ablations, canary, long rollout), end-to-end `DeltaBlock` + CUDA graphs + Prefill->Decode handoff, and `torch.compile(backend='inductor')`.


In [ ]:
!python tests/test_correctness.py


## 4. Training Benchmark (Token Mixer vs Baselines & E2E `DeltaBlock` + `torch.compile`)
Evaluates Part A (Pure Token Mixer vs PyTorch, FLA v1, FLA GatedDeltaNet-2, NVLabs GatedDeltaNet-2) and Part B (End-to-End `DeltaBlock` with MLP across `torch-eager`, `torch-compiled`, `kernel-eager`, `kernel-compiled`) and saves `assets/training_benchmark.png`.


In [ ]:
!python -u tests/bench_training.py \
    --dim 768 \
    --q-heads 8 \
    --kv-heads 8 \
    --batch-size 4 \
    --seq-lens 512,1024,2048,4096 \
    --e2e-batches 4,8,16 \
    --e2e-seq-len 1024 \
    --warmup 3 \
    --iters 10 \
    --save-plot assets/training_benchmark.png


In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("assets/training_benchmark.png"):
    display(Image("assets/training_benchmark.png"))


## 5. Inference Benchmark (Decode `L=1` & E2E Prefill+Generation vs Baselines & `torch.compile`)
Evaluates Part A (Single-step decode `L=1` across `B = 1..2048` vs PyTorch, FLA v1, FLA GatedDeltaNet-2, NVLabs GatedDeltaNet-2) and Part B (End-to-End `DeltaBlock` decode & Prefill+Generation rollout across `torch-eager`, `torch-compiled`, `kernel-eager`, `kernel-compiled`) and saves `assets/inference_benchmark.png`.


In [ ]:
!python -u tests/bench_inference.py \
    --dim 768 \
    --q-heads 8 \
    --kv-heads 8 \
    --batches 1,16,64,256,1024,2048 \
    --e2e-batches 16,64,256,1024 \
    --prefill-len 512 \
    --gen-steps 32 \
    --warmup 5 \
    --iters 20 \
    --save-plot assets/inference_benchmark.png


In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("assets/inference_benchmark.png"):
    display(Image("assets/inference_benchmark.png"))


## 6. Nsight Compute Profiling (`.ncu-rep` in `benchmark/`)
Collects hardware performance counters (`SM Throughput`, `DRAM Bandwidth`, `Achieved Occupancy`, `Register/Thread`) with **embedded CUDA/C++ source (`--import-source yes`)** into **9 separate reports** (each file isolates 1 kernel across multiple test cases):
- `kernel_chunk_wy_fwd`, `kernel_chunk_scan_fwd`, `kernel_output_combine_fwd`
- `kernel_output_combine_dq_dk_dv`, `kernel_output_combine_dc_inter_dS`, `kernel_output_combine_dc_total`
- `kernel_chunk_scan_bwd`, `kernel_chunk_wy_bwd`
  *(4 training cases: `short_b1=[128]`, `short_b16=[128]*16`, `skewed_b4=[257, 512, 1024, 2048]`, `long_b1=[4096]`)*
- `kernel_deltanet_infer`
  *(5 decode cases: `b1_gqa2`, `b16_gqa2`, `b64_gqa2`, `b256_gqa2`, `b1024_mha`)*

> Note: On some cloud containers without hardware counter permissions for `ncu`, the command may report `ERR_NVGPUCTRPERM`.


In [ ]:
%%bash
set -e
if ! command -v ncu &> /dev/null; then
    echo "[INFO] NVIDIA Nsight Compute (ncu) not found in PATH."
    exit 0
fi

NCU_SCRIPT='import sys, torch
from deltanet_v2 import custom_ops

kname = sys.argv[1]
dev = torch.device("cuda")
torch.manual_seed(42)

if kname == "kernel_deltanet_infer":
    cases = [
        ("b1_gqa2", 1, 8, 4),
        ("b16_gqa2", 16, 8, 4),
        ("b64_gqa2", 64, 8, 4),
        ("b256_gqa2", 256, 8, 4),
        ("b1024_mha", 1024, 8, 8),
    ]
    for _, B, hq, hkv in cases:
        past = (torch.randn(B, hkv, 64, 64, device=dev) * 0.1).half().contiguous()
        q = torch.randn(B, hq, 64, device=dev, dtype=torch.float16)
        k = torch.randn(B, hkv, 64, device=dev, dtype=torch.float16)
        v = torch.randn(B, hkv, 64, device=dev, dtype=torch.float16)
        b = torch.randn(B, hkv, 64, device=dev, dtype=torch.float16)
        w = torch.randn(B, hkv, 64, device=dev, dtype=torch.float16)
        a = torch.randn(B, hkv, 1, device=dev, dtype=torch.float16)
        g = torch.randn(B, hkv, 1, device=dev, dtype=torch.float16)
        custom_ops.deltanet_infer(past, q, k, v, b, w, a, g, hq, hkv, 64, 64)
else:
    train_cases = [
        ("short_b1", [128]),
        ("short_b16", [128] * 16),
        ("skewed_b4", [257, 512, 1024, 2048]),
        ("long_b1", [4096]),
    ]
    hq, hkv = 16, 4
    for _, lens in train_cases:
        B = len(lens)
        L = sum(lens)
        cu = torch.tensor([0] + torch.cumsum(torch.tensor(lens), 0).tolist(), dtype=torch.int32, device=dev)
        q = torch.randn(hq, L, 64, device=dev, dtype=torch.float16)
        k = torch.randn(hkv, L, 64, device=dev, dtype=torch.float16)
        v = torch.randn(hkv, L, 64, device=dev, dtype=torch.float16)
        w = torch.randn(hkv, L, 64, device=dev, dtype=torch.float16)
        b = torch.randn(hkv, L, 64, device=dev, dtype=torch.float16)
        a = torch.randn(hkv, L, 1, device=dev, dtype=torch.float16)
        g = torch.randn(hkv, L, 1, device=dev, dtype=torch.float16)
        past = (torch.randn(B, hkv, 64, 64, device=dev) * 0.1).half().contiguous()

        if kname == "kernel_chunk_wy_fwd":
            custom_ops.chunk_wy_fwd(k, v, b, w, g, a, cu, 64)
        elif kname == "kernel_chunk_scan_fwd":
            W, Y = custom_ops.chunk_wy_fwd(k, v, b, w, g, a, cu, 64)
            custom_ops.chunk_scan_fwd(W, Y, past, cu, 64)
        elif kname == "kernel_output_combine_fwd":
            W, Y = custom_ops.chunk_wy_fwd(k, v, b, w, g, a, cu, 64)
            S, _ = custom_ops.chunk_scan_fwd(W, Y, past, cu, 64)
            custom_ops.output_combine_fwd(q, k, v, b, w, g, a, S, cu, hq, hkv, 64, 64, 64)
        elif kname in ("kernel_output_combine_dq_dk_dv", "kernel_output_combine_dc_inter_dS", "kernel_output_combine_dc_total"):
            W, Y = custom_ops.chunk_wy_fwd(k, v, b, w, g, a, cu, 64)
            S, _ = custom_ops.chunk_scan_fwd(W, Y, past, cu, 64)
            dO = torch.randn_like(q)
            custom_ops.output_combine_bwd(dO, q, k, v, b, w, g, a, S, cu, hq, hkv, 64, 64, 64)
        elif kname == "kernel_chunk_scan_bwd":
            W, Y = custom_ops.chunk_wy_fwd(k, v, b, w, g, a, cu, 64)
            S, fs = custom_ops.chunk_scan_fwd(W, Y, past, cu, 64)
            dS = torch.randn_like(S)
            gfs = torch.randn_like(fs)
            custom_ops.chunk_scan_bwd(dS, W, S, cu, gfs, True, 64)
        elif kname == "kernel_chunk_wy_bwd":
            W, Y = custom_ops.chunk_wy_fwd(k, v, b, w, g, a, cu, 64)
            S, fs = custom_ops.chunk_scan_fwd(W, Y, past, cu, 64)
            dO = torch.randn_like(q)
            _, dk_ext, dv_ext, da_ext, dS = custom_ops.output_combine_bwd(dO, q, k, v, b, w, g, a, S, cu, hq, hkv, 64, 64, 64)
            dW, dY, _ = custom_ops.chunk_scan_bwd(dS, W, S, cu, torch.randn_like(fs), True, 64)
            custom_ops.chunk_wy_bwd(dW, dY, k, v, b, w, g, a, cu, dk_ext, dv_ext, da_ext, 64)

torch.cuda.synchronize()'

echo "$NCU_SCRIPT" > benchmark/_ncu_profile_target.py

KERNELS=(
    "kernel_chunk_wy_fwd"
    "kernel_chunk_scan_fwd"
    "kernel_output_combine_fwd"
    "kernel_output_combine_dq_dk_dv"
    "kernel_output_combine_dc_inter_dS"
    "kernel_output_combine_dc_total"
    "kernel_chunk_scan_bwd"
    "kernel_chunk_wy_bwd"
    "kernel_deltanet_infer"
)

for KNAME in "${KERNELS[@]}"; do
    echo "=== Profiling ${KNAME} (multi-case + embedded source) ==="
    ncu --set full --import-source yes --source-folders .,src,include -k "regex:${KNAME}" -f -o "benchmark/${KNAME}" python benchmark/_ncu_profile_target.py "${KNAME}" || true
done

rm -f benchmark/_ncu_profile_target.py
ls -lh benchmark/*.ncu-rep 2>/dev/null || echo "No .ncu-rep files generated." 
